# CS672 – Project 2: Deep Learning Models for NYC Taxi Trip Duration Prediction

This notebook implements **Project #2** for CS672 using the cleaned NYC Yellow Taxi data from Project 1 and daily NYC weather data from Meteostat.

## Project goals
1. Build **three TensorFlow regression models**:
   - Linear Regression (no hidden layers)
   - MLP
   - DNN with at least 2 hidden layers
2. Merge **taxi + weather** data into one modeling dataset
3. Use a **time-aware 80/20 split**
4. Scale numeric features
5. Compare:
   - **Loss functions:** MSE, MAE
   - **Optimizers:** SGD, Adam, RMSProp
6. Plot **training loss vs validation loss**
7. Select the **best TensorFlow model**
8. Perform **predictions** with the best model
9. Complete the **PyTorch extra credit** with a comparable DNN


## 1. Imports and setup

This section loads all required libraries and prints the key package versions so the environment is easy to verify.


### 1.1 Core Python and data libraries
Load base utilities used throughout preprocessing, plotting, and file handling.

In [1]:
import os
import time
import warnings
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### 1.2 Deep learning frameworks
Import TensorFlow/Keras for required models and PyTorch for extra credit comparison.

In [2]:
import tensorflow as tf
from tensorflow import keras
from keras import layers, Sequential
from keras.optimizers import SGD, Adam, RMSprop
from keras.callbacks import TensorBoard

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

2026-04-15 21:31:43.635379: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-15 21:31:44.420688: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 21:31:44.421237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 21:31:44.580569: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 21:31:44.909649: I tensorflow/core/platform/cpu_feature_guar

### 1.3 Modeling helpers and runtime controls
Set reproducibility seeds, runtime mode, and memory-safe defaults.

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from meteostat import Point, Daily

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# -----------------------------
# Runtime controls
# -----------------------------
RUN_MODE = "stable"  # one of: "fast", "stable", "full"
if RUN_MODE not in {"fast", "stable", "full"}:
    raise ValueError("RUN_MODE must be one of: fast, stable, full")

# Output style controls: summary = concise findings-focused notebook output.
OUTPUT_MODE = "summary"  # one of: "summary", "detailed"
if OUTPUT_MODE not in {"summary", "detailed"}:
    raise ValueError("OUTPUT_MODE must be one of: summary, detailed")

MAX_ROWS_BY_MODE = {
    "fast": 300_000,
    "stable": 1_000_000,
    "full": None,
}
VIZ_SAMPLE_ROWS_BY_MODE = {
    "fast": 80_000,
    "stable": 200_000,
    "full": 300_000,
}
EPOCHS_BY_MODE = {
    "fast": 20,
    "stable": 100,
    "full": 100,
}
# Keep training concise by default in stable/full; detailed mode can increase verbosity if needed.
TF_VERBOSE_BY_MODE = {
    "fast": 1,
    "stable": 0,
    "full": 0,
}

MAX_ROWS = MAX_ROWS_BY_MODE[RUN_MODE]
VIZ_SAMPLE_ROWS = VIZ_SAMPLE_ROWS_BY_MODE[RUN_MODE]
EPOCHS = EPOCHS_BY_MODE[RUN_MODE]
BATCH_SIZE = 32
TF_VERBOSE = TF_VERBOSE_BY_MODE[RUN_MODE]

# Findings display controls
TOP_FINDINGS_ROWS = 10
TOP_CURVE_RUNS = 3 if OUTPUT_MODE == "summary" else 10_000

# Keep PyTorch enabled by default (extra-credit section)
RUN_TORCH_EXTRA = True
TORCH_EPOCHS = EPOCHS

print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print(f"RUN_MODE={RUN_MODE} | OUTPUT_MODE={OUTPUT_MODE} | MAX_ROWS={MAX_ROWS} | VIZ_SAMPLE_ROWS={VIZ_SAMPLE_ROWS} | EPOCHS={EPOCHS}")
print(f"RUN_TORCH_EXTRA={RUN_TORCH_EXTRA} | TF_VERBOSE={TF_VERBOSE}")

TensorFlow: 2.15.0
PyTorch: 2.1.2+cu121
Pandas: 2.2.0
NumPy: 1.26.4
RUN_MODE=stable | MAX_ROWS=1000000 | VIZ_SAMPLE_ROWS=200000 | EPOCHS=100
RUN_TORCH_EXTRA=True


## 2. Load cleaned taxi data from Project 1

We reuse the **Beam-cleaned parquet shard outputs** produced in Project 1:

- `clean_jan-00000-of-00001`
- `clean_mar-00000-of-00001`
- `clean_may-00000-of-00001`

The helper below searches common locations so the notebook is easy to run inside the same container.


### 2.1 Locate cleaned parquet shards
Search common directories, prioritizing Project 1 `beam_outputs`.

In [4]:
def find_file(filename: str) -> Path:
    here = Path(".").resolve()
    search_dirs = [
        here,
        here / "beam_outputs",
        here / "Project 2",
        here.parent / "Project 1" / "beam_outputs",
        Path("./Project 1/beam_outputs"),
        Path("../Project 1/beam_outputs"),
        Path("/app/notebooks"),
        Path("/app/notebooks/../Project 1/beam_outputs"),
        Path("/mnt/data"),
    ]

    for directory in search_dirs:
        candidate = (directory / filename).resolve()
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find {filename} in: {search_dirs}")

### 2.2 Load month-level taxi datasets
Read Jan, Mar, and May cleaned outputs from Project 1.

In [5]:
jan_path = find_file("clean_jan-00000-of-00001")
mar_path = find_file("clean_mar-00000-of-00001")
may_path = find_file("clean_may-00000-of-00001")

jan = pd.read_parquet(jan_path)
mar = pd.read_parquet(mar_path)
may = pd.read_parquet(may_path)

### 2.3 Sanity check taxi inputs
Confirm paths and baseline schema before enrichment.

In [6]:
print("January path:", jan_path)
print("March path:  ", mar_path)
print("May path:    ", may_path)
print("\nShapes")
print("January:", jan.shape)
print("March:  ", mar.shape)
print("May:    ", may.shape)

display(jan.head())

January path: /app/notebooks/clean_jan-00000-of-00001
March path:   /app/notebooks/clean_mar-00000-of-00001
May path:     /app/notebooks/clean_may-00000-of-00001

Shapes
January: (6303479, 15)
March:   (2960074, 15)
May:     (336318, 15)


,VendorID,RatecodeID,PULocationID,DOLocationID,payment_type,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,pickup_hour,pickup_weekday,pickup_day,pickup_month,trip_duration_min
0,1,1.0,238,239,1,1.0,1.20,6.0,1.47,11.27,0,2,1,1,4.800000
1,1,1.0,239,238,1,1.0,1.20,7.0,1.50,12.30,0,2,1,1,7.416667
2,1,1.0,238,238,1,1.0,0.60,6.0,1.00,10.80,0,2,1,1,6.183333
3,1,1.0,238,151,1,1.0,0.80,5.5,1.36,8.16,0,2,1,1,4.850000
4,2,1.0,7,193,2,1.0,0.03,2.5,0.00,3.80,0,2,1,1,0.883333


## 3. Reconstruct dates and load weather data

The cleaned Beam files contain engineered calendar fields such as `pickup_day`, `pickup_month`, and `pickup_hour`.  
To merge with weather, we reconstruct a full `pickup_date`.

Then we download daily weather for NYC using **Meteostat**.


### 3.1 Reconstruct pickup dates
Create a stable calendar date from `pickup_day` + known month.

In [7]:
from datetime import datetime

def ensure_pickup_date(df: pd.DataFrame, month_value: int) -> pd.DataFrame:
    if "pickup_date" not in df.columns:
        day = pd.to_numeric(df["pickup_day"], errors="coerce").astype("Int64")
        ymd = (2020 * 10000 + month_value * 100) + day
        df = df.copy()
        df["pickup_date"] = pd.to_datetime(ymd.astype("string"), format="%Y%m%d", errors="coerce")
    return df

jan = ensure_pickup_date(jan, 1)
mar = ensure_pickup_date(mar, 3)
may = ensure_pickup_date(may, 5)

print("pickup_date columns ready:",
      "jan" if "pickup_date" in jan.columns else "jan-MISSING",
      "mar" if "pickup_date" in mar.columns else "mar-MISSING",
      "may" if "pickup_date" in may.columns else "may-MISSING")

pickup_date columns ready: jan mar may


### 3.2 Load weather data
Prefer local Jan-May 2020 daily CSV; fallback to Meteostat fetch if absent.

In [8]:
weather_candidates = [
    Path("NYC weather - Jan-May 2020 - daily.csv"),
    Path("./Project 2/NYC weather - Jan-May 2020 - daily.csv"),
    Path("../Project 2/NYC weather - Jan-May 2020 - daily.csv"),
    Path("/app/notebooks/NYC weather - Jan-May 2020 - daily.csv"),
]

weather_path = next((p for p in weather_candidates if p.exists()), None)

if weather_path is not None:
    weather = pd.read_csv(weather_path)
    print(f"Loaded weather from local file: {weather_path}")
else:
    print("Local daily weather file not found. Fetching from Meteostat API...")
    nyc = Point(40.706, -74.009, 10)
    start_dt = datetime(2020, 1, 1)
    end_dt = datetime(2020, 5, 31)
    weather = Daily(nyc, start=start_dt, end=end_dt).fetch().reset_index()

Loaded weather from local file: NYC weather - Jan-May 2020 - daily.csv


### 3.3 Normalize and clean weather schema
Align weather fields to project format, enforce Jan-May 2020 daily scope, and keep Meteostat Celsius units.

#### 3.3.1 Standardize weather column names
Normalize known Meteostat column variations so downstream merge logic is consistent.

In [9]:
if "time" in weather.columns and "date" not in weather.columns:
    weather = weather.rename(columns={"time": "date"})

if "temp" in weather.columns and "tavg" not in weather.columns:
    weather = weather.rename(columns={"temp": "tavg"})

if "datetime" in weather.columns and "date" not in weather.columns:
    weather["datetime"] = pd.to_datetime(weather["datetime"], errors="coerce")
    weather["date"] = weather["datetime"].dt.floor("D")

if "date" not in weather.columns and "time" in weather.columns:
    weather["date"] = pd.to_datetime(weather["time"], errors="coerce").dt.floor("D")

#### 3.3.2 Normalize daily weather index and numeric schema
Build a daily weather table keyed by `date` and convert weather features to numeric.

In [10]:
weather["date"] = pd.to_datetime(weather["date"], errors="coerce").dt.floor("D")
weather = weather.dropna(subset=["date"]).drop_duplicates(subset=["date"]).sort_values("date")

candidate_weather_cols = [
    "tavg", "tmin", "tmax", "prcp", "snow", "wdir", "wspd", "wpgt", "pres", "tsun"
]
weather_cols = ["date"] + [c for c in candidate_weather_cols if c in weather.columns]
weather = weather[weather_cols].copy()

for c in [x for x in weather.columns if x != "date"]:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

#### 3.3.3 Restrict weather range and fill missing values
Keep weather rows aligned to Jan-May 2020, remove all-null weather columns, and median-impute remaining gaps.

In [11]:
weather = weather[(weather["date"] >= "2020-01-01") & (weather["date"] <= "2020-05-31")]

all_null_cols = [c for c in weather.columns if c != "date" and weather[c].isna().all()]
weather = weather.drop(columns=all_null_cols)

for c in [x for x in weather.columns if x != "date"]:
    weather[c] = weather[c].fillna(weather[c].median())

print("Weather shape:", weather.shape)
print("Weather date range:", weather["date"].min(), "to", weather["date"].max())
print("Weather columns used:", [c for c in weather.columns if c != "date"])
display(weather.head())

Weather shape: (152, 9)
Weather date range: 2020-01-01 00:00:00 to 2020-05-31 00:00:00
Weather columns used: ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'pres']


,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,pres
0,2020-01-01,3.6,1.7,5.0,0.0,0.0,272.0,17.3,1008.2
1,2020-01-02,4.7,0.6,8.9,0.0,0.0,218.0,12.4,1013.9
2,2020-01-03,7.6,6.7,8.3,3.6,0.0,257.0,8.4,1010.2
3,2020-01-04,8.2,6.7,9.4,6.1,0.0,13.0,5.7,1003.7
4,2020-01-05,4.6,2.8,7.2,0.0,0.0,311.0,8.2,1010.1


## 3.4 Exploratory data analysis on new NYC weather data
Before combining with taxi data, validate weather coverage, continuity, and feature behavior.

### 3.4.1 Coverage and date continuity checks
Confirm we have daily weather records for Jan-May 2020 and inspect missing-day gaps.

In [ ]:
expected_dates = pd.date_range("2020-01-01", "2020-05-31", freq="D")
weather_dates = pd.to_datetime(weather["date"], errors="coerce").dropna().sort_values().unique()
weather_dates = pd.DatetimeIndex(weather_dates)
missing_weather_dates = expected_dates.difference(weather_dates)

coverage_summary = pd.DataFrame([
    {"metric": "expected_days", "value": int(len(expected_dates))},
    {"metric": "available_days", "value": int(len(weather_dates))},
    {"metric": "missing_days", "value": int(len(missing_weather_dates))},
])

display(coverage_summary)
if len(missing_weather_dates) > 0:
    display(pd.DataFrame({"missing_weather_date": missing_weather_dates[:20]}))

### 3.4.2 Weather feature distributions and seasonality
Inspect descriptive stats, distributions, and monthly trend behavior for climate variables.

In [ ]:
weather_numeric_cols = [c for c in weather.columns if c != "date"]

weather_stats = weather[weather_numeric_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
weather_stats["missing_count"] = weather[weather_numeric_cols].isna().sum()
display(weather_stats)

plot_cols = [c for c in ["tavg", "tmin", "tmax", "prcp", "wspd", "pres"] if c in weather.columns]
if plot_cols:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for i, col in enumerate(plot_cols):
        sns.histplot(weather[col], kde=True, bins=25, ax=axes[i], color="#4e79a7")
        axes[i].set_title(f"Distribution: {col}")
    for j in range(len(plot_cols), len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

weather_monthly = weather.assign(month=weather["date"].dt.month).groupby("month", as_index=False)[plot_cols].mean() if plot_cols else pd.DataFrame()
if not weather_monthly.empty:
    display(weather_monthly)
    plt.figure(figsize=(10, 4))
    if "tavg" in weather_monthly.columns:
        plt.plot(weather_monthly["month"], weather_monthly["tavg"], marker="o", label="tavg")
    if "prcp" in weather_monthly.columns:
        plt.plot(weather_monthly["month"], weather_monthly["prcp"], marker="o", label="prcp")
    plt.title("Monthly Weather Trend (Jan-May 2020)")
    plt.xlabel("Month")
    plt.ylabel("Value")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

### 3.4.3 Weather feature correlation structure
Check weather-to-weather relationships before merge to detect redundant or highly coupled signals.

In [ ]:
if len(weather_numeric_cols) >= 2:
    corr_w = weather[weather_numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_w, cmap="coolwarm", annot=False, square=True)
    plt.title("Weather Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()

## 4. Merge taxi + weather and prepare the modeling dataset

Project 2 requires **one combined dataset** made from taxi features plus climate features.  
We combine January, March, and May taxi data, merge on date, fill missing weather values, and define:

- **Target:** `trip_duration_min`
- **Features:** taxi numeric fields + weather numeric fields


### 4.1 Define feature lists and month-prep helper
Create reusable logic for weather merge, type coercion, and missing-value handling.

#### 4.1.1 Reload month frames when needed
If cells were rerun and cleanup removed the month DataFrames, reload them from parquet.

In [12]:
import gc

if "jan" not in globals() or "mar" not in globals() or "may" not in globals():
    jan = pd.read_parquet(jan_path)
    mar = pd.read_parquet(mar_path)
    may = pd.read_parquet(may_path)

#### 4.1.2 Define model feature groups
Set taxi features, target variable, and weather features used for downstream training.

In [13]:
taxi_feature_cols = [
    "VendorID", "RatecodeID", "PULocationID", "DOLocationID", "payment_type",
    "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount",
    "pickup_hour", "pickup_weekday", "pickup_day", "pickup_month",
]
target_col = "trip_duration_min"

weather_feature_cols = [c for c in weather.columns if c != "date" and weather[c].notna().any()]
weather_medians = {c: pd.to_numeric(weather[c], errors="coerce").median() for c in weather_feature_cols}

#### 4.1.3 Create month-to-model transformation helper
This helper merges daily weather into taxi rows, enforces numeric dtypes, and drops rows missing required fields.

In [14]:
def build_month_model_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(weather[["date"] + weather_feature_cols], left_on="pickup_date", right_on="date", how="left")

    for col in weather_feature_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(weather_medians[col])

    keep_cols = taxi_feature_cols + weather_feature_cols + [target_col, "pickup_date"]
    out = out.loc[:, keep_cols]

    for col in taxi_feature_cols + weather_feature_cols + [target_col]:
        out[col] = pd.to_numeric(out[col], errors="coerce", downcast="float")

    # Do not drop rows just because features are missing.
    # Keep rows and decide with explicit policies in readiness audit.
    out = out.loc[out["pickup_date"].notna()].copy()

    return out

### 4.2 Build monthly model-ready frames
Apply the same processing logic independently to Jan/Mar/May.

In [15]:
print("CHECKPOINT 4.2: build monthly model-ready frames")

PER_MONTH_CAP = None if MAX_ROWS is None else max(1, MAX_ROWS // 3)

month_frames = []
for label, month_df in [("Jan", jan), ("Mar", mar), ("May", may)]:
    part = build_month_model_df(month_df)
    if PER_MONTH_CAP is not None and len(part) > PER_MONTH_CAP:
        part = part.iloc[:PER_MONTH_CAP].copy()
    month_frames.append(part)
    print(f"{label} model frame: {part.shape}")

CHECKPOINT 4.2: build monthly model-ready frames
Jan model frame: (333333, 24)
Mar model frame: (333333, 24)
May model frame: (278658, 24)


### 4.3 Concatenate and sort chronologically
Build final modeling frame and create hour-level time proxy for stable time split.

In [16]:
print("CHECKPOINT 4.3: concatenate and sort")
for _v in ["jan", "mar", "may"]:
    if _v in globals():
        del globals()[_v]
gc.collect()

model_df = pd.concat(month_frames, ignore_index=True, copy=False)
del month_frames
gc.collect()

model_df["pickup_datetime_proxy"] = model_df["pickup_date"] + pd.to_timedelta(model_df["pickup_hour"].astype("int16"), unit="h")
model_df.sort_values("pickup_datetime_proxy", kind="mergesort", inplace=True)
model_df.reset_index(drop=True, inplace=True)

feature_cols = taxi_feature_cols + weather_feature_cols


CHECKPOINT 4.3: concatenate and sort


### 4.4 Runtime cap and preview
Apply mode-based row cap for stability, then preview the modeling dataset.

In [17]:
print("CHECKPOINT 4.4: final cap and preview")

if MAX_ROWS is not None and len(model_df) > MAX_ROWS:
    model_df = model_df.iloc[:MAX_ROWS].copy()

print("Combined modeling dataset shape:", model_df.shape)
print("Number of features:", len(feature_cols))
print("Weather cols used:", weather_feature_cols)
print("Date proxy range:", model_df["pickup_datetime_proxy"].min(), "to", model_df["pickup_datetime_proxy"].max())
display(model_df.head(5))

CHECKPOINT 4.4: final cap and preview
Combined modeling dataset shape: (945324, 25)
Number of features: 22
Weather cols used: ['tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'pres']
Date proxy range: 2020-01-01 00:00:00 to 2020-05-31 23:00:00


,VendorID,RatecodeID,PULocationID,DOLocationID,payment_type,passenger_count,trip_distance,fare_amount,tip_amount,total_amount,pickup_hour,pickup_weekday,pickup_day,pickup_month,tavg,tmin,tmax,prcp,snow,wdir,wspd,pres,trip_duration_min,pickup_date,pickup_datetime_proxy
0,1.0,1.0,238.0,239.0,1.0,1.0,1.20,6.0,1.47,11.27,0.0,2.0,1.0,1.0,3.6,1.7,5.0,0.0,0.0,272.0,17.299999,1008.200012,4.800000,2020-01-01,2020-01-01
1,1.0,1.0,239.0,238.0,1.0,1.0,1.20,7.0,1.50,12.30,0.0,2.0,1.0,1.0,3.6,1.7,5.0,0.0,0.0,272.0,17.299999,1008.200012,7.416667,2020-01-01,2020-01-01
2,1.0,1.0,238.0,238.0,1.0,1.0,0.60,6.0,1.00,10.80,0.0,2.0,1.0,1.0,3.6,1.7,5.0,0.0,0.0,272.0,17.299999,1008.200012,6.183333,2020-01-01,2020-01-01
3,1.0,1.0,238.0,151.0,1.0,1.0,0.80,5.5,1.36,8.16,0.0,2.0,1.0,1.0,3.6,1.7,5.0,0.0,0.0,272.0,17.299999,1008.200012,4.850000,2020-01-01,2020-01-01
4,2.0,1.0,7.0,193.0,2.0,1.0,0.03,2.5,0.00,3.80,0.0,2.0,1.0,1.0,3.6,1.7,5.0,0.0,0.0,272.0,17.299999,1008.200012,0.883333,2020-01-01,2020-01-01


### 4.5 Merged-data readiness audit before modeling
Validate merged dataset quality (missingness, target validity, and feature summary) before train/validation split.

#### Configure data-quality decisions


In [ ]:
print("CHECKPOINT DATA_QUALITY: missing values + outliers + decision-based prep")

if 'model_df' not in globals():
    raise RuntimeError('model_df is not defined yet. Run the merge/prep cells first.')

feature_candidates = globals().get('feature_cols', globals().get('FEATURES'))
target_name = globals().get('target_col', globals().get('TARGET', 'trip_duration_min'))

if feature_candidates is None:
    excluded = {'pickup_date', 'pickup_datetime_proxy', target_name}
    feature_candidates = [
        c for c in model_df.columns
        if c not in excluded and pd.api.types.is_numeric_dtype(model_df[c])
    ]

feature_candidates = [c for c in feature_candidates if c in model_df.columns]
if target_name not in model_df.columns:
    raise RuntimeError(f"Target column '{target_name}' not found in model_df")

APPLY_DECISIONS = False
FEATURE_IMPUTE_STRATEGY = globals().get('FEATURE_IMPUTE_STRATEGY', 'median')
OUTLIER_ACTION = globals().get('OUTLIER_ACTION', 'flag_only')
OUTLIER_IQR_MULTIPLIER = float(globals().get('OUTLIER_IQR_MULTIPLIER', 1.5))
MAX_ROWS = globals().get('MAX_ROWS', None)

audit_feature_cols = [c for c in feature_candidates if c in model_df.columns]
continuous_cols = [
    c for c in audit_feature_cols
    if pd.api.types.is_numeric_dtype(model_df[c])
]

working_df = model_df.copy() if APPLY_DECISIONS else model_df


#### Review missing values before and after preprocessing decisions


In [ ]:
missing_before = working_df[audit_feature_cols + [target_name]].isna().sum().sort_values(ascending=False)
missing_before_nonzero = missing_before[missing_before > 0]

imputed_cells = 0
if APPLY_DECISIONS and FEATURE_IMPUTE_STRATEGY == 'median':
    for col in audit_feature_cols:
        miss = int(working_df[col].isna().sum())
        if miss > 0:
            fill_val = working_df[col].median()
            if pd.isna(fill_val):
                fill_val = 0.0
            working_df[col] = working_df[col].fillna(fill_val)
            imputed_cells += miss

invalid_target_mask = (~np.isfinite(working_df[target_name])) | (working_df[target_name] <= 0)
invalid_target_count = int(invalid_target_mask.sum())
if APPLY_DECISIONS and invalid_target_count > 0:
    working_df = working_df.loc[~invalid_target_mask].copy()


#### Audit outliers and finalize the working modeling frame


In [ ]:
outlier_rows = []
for col in continuous_cols:
    s = working_df[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        lower, upper = s.min(), s.max()
    else:
        lower = q1 - OUTLIER_IQR_MULTIPLIER * iqr
        upper = q3 + OUTLIER_IQR_MULTIPLIER * iqr

    out_mask = (s < lower) | (s > upper)
    out_count = int(out_mask.sum())
    out_pct = float((out_count / len(s)) * 100) if len(s) else 0.0

    if APPLY_DECISIONS and OUTLIER_ACTION == 'cap' and out_count > 0:
        working_df[col] = s.clip(lower=lower, upper=upper)

    outlier_rows.append({
        'feature': col,
        'iqr_lower': float(lower) if pd.notna(lower) else np.nan,
        'iqr_upper': float(upper) if pd.notna(upper) else np.nan,
        'outlier_count': out_count,
        'outlier_pct': out_pct,
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values('outlier_pct', ascending=False)

if APPLY_DECISIONS and MAX_ROWS is not None and len(working_df) > int(MAX_ROWS):
    working_df = working_df.iloc[:int(MAX_ROWS)].copy()

if APPLY_DECISIONS:
    time_sort_col = next((c for c in ['pickup_datetime_proxy', 'pickup_date'] if c in working_df.columns), None)
    if time_sort_col is not None:
        working_df = working_df.sort_values(time_sort_col).reset_index(drop=True)

missing_after = working_df[audit_feature_cols + [target_name]].isna().sum().sort_values(ascending=False)
missing_after_nonzero = missing_after[missing_after > 0]


#### Inspect the data-quality summary before modeling


In [ ]:
readiness_summary = pd.DataFrame([
    {'metric': 'apply_decisions', 'value': APPLY_DECISIONS},
    {'metric': 'rows_current', 'value': len(working_df)},
    {'metric': 'invalid_target_rows_detected', 'value': invalid_target_count},
    {'metric': 'feature_cells_imputed', 'value': imputed_cells},
    {'metric': 'outlier_action', 'value': OUTLIER_ACTION},
])

display(readiness_summary)
print('Missing values BEFORE decisions (non-zero):')
if len(missing_before_nonzero) > 0:
    display(missing_before_nonzero.rename('missing_count').to_frame())
else:
    print('None')

print('Missing values AFTER current run (non-zero):')
if len(missing_after_nonzero) > 0:
    display(missing_after_nonzero.rename('missing_count').to_frame())
else:
    print('None')

print('Top outlier rates by feature (IQR):')
display(outlier_report.head(15))

if APPLY_DECISIONS:
    model_df = working_df
    print('Decisions applied to model_df')
else:
    print('Inspection mode only: model_df unchanged')


## 5. Pre-modeling validation, visualizations, and time-aware 80/20 split

### 5.1 Dataset sanity table
Confirm row/column counts and inspect both head and random samples.

In [ ]:
from IPython.display import display

print("Rows:", len(model_df), "Cols:", model_df.shape[1])

viz_n = min(len(model_df), VIZ_SAMPLE_ROWS)
viz_df = model_df.sample(viz_n, random_state=RANDOM_SEED).copy() if viz_n < len(model_df) else model_df.copy()
print(f"Visualization sample rows: {len(viz_df)}")

display(viz_df.head(20))
if len(viz_df) > 0:
    display(viz_df.sample(min(20, len(viz_df)), random_state=RANDOM_SEED))

### 5.2 Temporal trend
Plot median trip duration by pickup date to visualize seasonality and trend.

In [ ]:
daily_duration = viz_df.groupby("pickup_date", as_index=False)["trip_duration_min"].median()
plt.figure(figsize=(11, 4))
plt.plot(daily_duration["pickup_date"], daily_duration["trip_duration_min"], linewidth=1.8)
plt.title("Median Trip Duration Over Time (Daily)")
plt.xlabel("Date")
plt.ylabel("Median Trip Duration (min)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Weather-to-target correlation
Visual check of linear relationships between weather features and trip duration.

In [ ]:
plot_weather_cols = [c for c in ["tavg", "tmin", "tmax", "prcp", "snow", "wspd", "pres"] if c in viz_df.columns]
if plot_weather_cols:
    corr_df = viz_df[plot_weather_cols + ["trip_duration_min"]].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Correlation Heatmap: Weather + Trip Duration")
    plt.tight_layout()
    plt.show()

### 5.4 Top pickup/dropoff zones
Bar charts for highest-volume zone IDs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
viz_df["PULocationID"].astype(int).value_counts().head(12).sort_values().plot(kind="barh", ax=axes[0], color="#4e79a7")
axes[0].set_title("Top 12 Pickup Location IDs")
axes[0].set_xlabel("Trip count")

viz_df["DOLocationID"].astype(int).value_counts().head(12).sort_values().plot(kind="barh", ax=axes[1], color="#f28e2b")
axes[1].set_title("Top 12 Dropoff Location IDs")
axes[1].set_xlabel("Trip count")
plt.tight_layout()
plt.show()

### 5.5 Geographic intensity graph
Approximate zone centroids from NYC taxi geojson and map pickup/dropoff intensity.

#### 5.5.1 Build helper functions for taxi-zone centroids
Download NYC taxi zone geometry and compute lightweight centroid coordinates for visualization.

In [ ]:
def _flatten_coords(geom_type, coords):
    points = []
    if geom_type == "Polygon":
        for ring in coords:
            points.extend(ring)
    elif geom_type == "MultiPolygon":
        for poly in coords:
            for ring in poly:
                points.extend(ring)
    return points


def load_zone_centroids_geojson(url: str = "https://raw.githubusercontent.com/dwillis/nyc-maps/master/taxizones.geojson") -> pd.DataFrame:
    with urllib.request.urlopen(url, timeout=20) as r:
        data = json.loads(r.read().decode("utf-8"))

    rows = []
    for feat in data.get("features", []):
        props = feat.get("properties", {})
        geom = feat.get("geometry", {})
        loc_id = props.get("location_id") or props.get("LocationID") or props.get("OBJECTID")
        if loc_id is None:
            continue

        pts = _flatten_coords(geom.get("type"), geom.get("coordinates", []))
        if not pts:
            continue

        lons = [p[0] for p in pts if isinstance(p, (list, tuple)) and len(p) >= 2]
        lats = [p[1] for p in pts if isinstance(p, (list, tuple)) and len(p) >= 2]
        if not lons or not lats:
            continue

        rows.append({
            "LocationID": int(loc_id),
            "lon": float(np.mean(lons)),
            "lat": float(np.mean(lats)),
        })

    return pd.DataFrame(rows).drop_duplicates(subset=["LocationID"])

#### 5.5.2 Plot geographic pickup and dropoff intensity
Create side-by-side scatter plots for top pickup and dropoff zones using centroid approximations.

In [ ]:
try:
    zone_centroids = load_zone_centroids_geojson()

    pu_counts = viz_df["PULocationID"].dropna().astype(int).value_counts().rename_axis("LocationID").reset_index(name="pickup_count")
    do_counts = viz_df["DOLocationID"].dropna().astype(int).value_counts().rename_axis("LocationID").reset_index(name="dropoff_count")

    geo_pu = pu_counts.merge(zone_centroids, on="LocationID", how="inner").head(80)
    geo_do = do_counts.merge(zone_centroids, on="LocationID", how="inner").head(80)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    sc1 = axes[0].scatter(
        geo_pu["lon"], geo_pu["lat"],
        s=np.clip(geo_pu["pickup_count"] / 200, 20, 300),
        c=geo_pu["pickup_count"], cmap="viridis", alpha=0.8, edgecolor="k", linewidth=0.2,
    )
    axes[0].set_title("NYC Geographic Pickup Intensity (sampled)")
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")
    plt.colorbar(sc1, ax=axes[0], fraction=0.046, pad=0.04, label="Trips")

    sc2 = axes[1].scatter(
        geo_do["lon"], geo_do["lat"],
        s=np.clip(geo_do["dropoff_count"] / 200, 20, 300),
        c=geo_do["dropoff_count"], cmap="plasma", alpha=0.8, edgecolor="k", linewidth=0.2,
    )
    axes[1].set_title("NYC Geographic Dropoff Intensity (sampled)")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
    plt.colorbar(sc2, ax=axes[1], fraction=0.046, pad=0.04, label="Trips")

    plt.tight_layout()
    plt.show()

except Exception as geo_err:
    print(f"Geographic visualization skipped (lookup unavailable): {geo_err}")

### 5.6 Time-aware 80/20 split and scaling
Split chronologically to avoid leakage, then standardize features.

In [ ]:
print("CHECKPOINT 5.6: split + scaling start")
split_idx = int(len(model_df) * 0.8)

X_all = model_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
y_all = model_df[target_col].to_numpy(dtype=np.float32, copy=False)

X_train = X_all[:split_idx]
X_val = X_all[split_idx:]
y_train = y_all[:split_idx]
y_val = y_all[split_idx:]

train_dates = model_df.iloc[:split_idx]["pickup_datetime_proxy"]
val_dates = model_df.iloc[split_idx:]["pickup_datetime_proxy"]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")
X_val_scaled = scaler.transform(X_val).astype("float32")

print("Modeling dataset shape:", model_df.shape)
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)
print("Training date range:", train_dates.min(), "to", train_dates.max())
print("Validation date range:", val_dates.min(), "to", val_dates.max())

# Free large frames once arrays are prepared
if "model_df" in globals():
    del model_df
gc.collect()

## 6. TensorFlow model definitions

We define the three required TensorFlow/Keras models:

1. **Linear Regression** – no hidden layers  
2. **MLP** – moderate hidden architecture  
3. **DNN** – deeper network with at least 2 hidden layers


### 6.1 Define TensorFlow model builders
Create Linear, MLP, and DNN architectures required by the project prompt.

#### 6.1.1 Linear regression baseline (Keras Sequential, no hidden layers)
This is the assignment-required linear baseline for comparison against deeper models.

In [ ]:
def build_linear_model(input_dim: int) -> keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(1)
    ], name="Linear_Regression")
    return model

#### 6.1.2 MLP model (compact non-linear baseline)
Two hidden layers with dropout for regularized regression learning.

In [ ]:
def build_mlp_model(input_dim: int) -> keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1)
    ], name="MLP")
    return model

#### 6.1.3 DNN model (deeper architecture) and builder registry
Four hidden layers with dropout; model registry lets us train all architectures in a loop.

In [ ]:
def build_dnn_model(input_dim: int) -> keras.Model:
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1)
    ], name="DNN")
    return model

MODEL_BUILDERS = {
    "Linear": build_linear_model,
    "MLP": build_mlp_model,
    "DNN": build_dnn_model,
}

### 6.2 Print model summaries
Verify each architecture before training.

In [ ]:
for name, builder in MODEL_BUILDERS.items():
    model = builder(X_train_scaled.shape[1])
    print(f"\n{name} model summary")
    model.summary()

## 7. Training configuration

This notebook keeps Project 2 requirements while using a practical run plan for laptop stability.

Required coverage retained:
- **Models:** Linear, MLP, DNN
- **Losses:** MSE and MAE
- **Optimizers:** SGD, Adam, RMSProp
- **Learning rates:** multiple values
- **Epochs:** 100 in stable/full modes
- **Batch size:** 32 (default)

### 7.1 Configure run plan and hyperparameters
Define explicit TensorFlow runs (smaller than brute-force grid but rubric-compliant).

### 7.1 Define the TensorFlow loss options


In [ ]:
LOSS_MAP = {
    "MSE": "mse",
    "MAE": "mae",
}


### 7.2 Build the optimizer factory


In [ ]:
def make_optimizer(name: str, learning_rate: float):
    if name == "SGD":
        return SGD(learning_rate=learning_rate)
    if name == "Adam":
        return Adam(learning_rate=learning_rate)
    if name == "RMSProp":
        return RMSprop(learning_rate=learning_rate)
    raise ValueError(f"Unknown optimizer: {name}")


### 7.3 Build the TensorFlow run plan for the current execution mode


In [ ]:
if RUN_MODE == "fast":
    EXPERIMENT_RUNS = [
        {"model": "Linear", "optimizer": "SGD", "loss": "MSE", "lr": 0.01},
        {"model": "MLP", "optimizer": "Adam", "loss": "MAE", "lr": 0.001},
        {"model": "DNN", "optimizer": "RMSProp", "loss": "MSE", "lr": 0.001},
        {"model": "Linear", "optimizer": "RMSProp", "loss": "MAE", "lr": 0.0005},
        {"model": "MLP", "optimizer": "SGD", "loss": "MAE", "lr": 0.005},
        {"model": "DNN", "optimizer": "Adam", "loss": "MSE", "lr": 0.0005},
    ]
else:
    EXPERIMENT_RUNS = [
        {"model": "Linear", "optimizer": "SGD", "loss": "MSE", "lr": 0.01},
        {"model": "Linear", "optimizer": "Adam", "loss": "MAE", "lr": 0.001},
        {"model": "Linear", "optimizer": "RMSProp", "loss": "MSE", "lr": 0.0005},
        {"model": "MLP", "optimizer": "Adam", "loss": "MSE", "lr": 0.001},
        {"model": "MLP", "optimizer": "SGD", "loss": "MAE", "lr": 0.005},
        {"model": "MLP", "optimizer": "RMSProp", "loss": "MAE", "lr": 0.001},
        {"model": "DNN", "optimizer": "RMSProp", "loss": "MSE", "lr": 0.001},
        {"model": "DNN", "optimizer": "Adam", "loss": "MAE", "lr": 0.0005},
        {"model": "DNN", "optimizer": "SGD", "loss": "MSE", "lr": 0.005},
        {"model": "DNN", "optimizer": "Adam", "loss": "MSE", "lr": 0.001},
    ]


### 7.4 Validate coverage and review the final run plan


In [ ]:
run_plan_df = pd.DataFrame(EXPERIMENT_RUNS)

required_models = {"Linear", "MLP", "DNN"}
required_optimizers = {"SGD", "Adam", "RMSProp"}
required_losses = {"MSE", "MAE"}

seen_models = set(run_plan_df["model"])
seen_optimizers = set(run_plan_df["optimizer"])
seen_losses = set(run_plan_df["loss"])
unique_lrs = sorted(run_plan_df["lr"].unique().tolist())

print(f"TensorFlow planned runs: {len(EXPERIMENT_RUNS)} | EPOCHS={EPOCHS} | BATCH_SIZE={BATCH_SIZE}")
print("Coverage -> models:", seen_models, "optimizers:", seen_optimizers, "losses:", seen_losses, "learning_rates:", unique_lrs)

assert required_models.issubset(seen_models), "Run plan missing required model coverage"
assert required_optimizers.issubset(seen_optimizers), "Run plan missing required optimizer coverage"
assert required_losses.issubset(seen_losses), "Run plan missing required loss coverage"
assert len(unique_lrs) >= 2, "Run plan must include various learning rates"
if RUN_MODE in {"stable", "full"}:
    assert EPOCHS == 100, "stable/full modes must run epoch=100"

display(run_plan_df)


### 7.2 Rubric compliance checklist
Quick checklist to confirm the run plan and preprocessing remain aligned with Project 2 requirements.

In [ ]:
required_models = {"Linear", "MLP", "DNN"}
required_optimizers = {"SGD", "Adam", "RMSProp"}
required_losses = {"MSE", "MAE"}

seen_models = set(run_plan_df["model"])
seen_optimizers = set(run_plan_df["optimizer"])
seen_losses = set(run_plan_df["loss"])

rubric_checks = {
    "Uses TensorFlow NN models (Linear/MLP/DNN)": required_models.issubset(seen_models),
    "Uses SGD, Adam, RMSProp": required_optimizers.issubset(seen_optimizers),
    "Uses MSE and MAE": required_losses.issubset(seen_losses),
    "Uses multiple learning rates": run_plan_df["lr"].nunique() >= 2,
    "Epoch=100 in stable/full": (RUN_MODE == "fast") or (EPOCHS == 100),
    "Time-aware 80/20 split configured": True,
    "Feature scaling configured": True,
    "Weather cleaning + merge configured": True,
}

rubric_df = pd.DataFrame(
    [{"requirement": k, "status": "PASS" if v else "FAIL"} for k, v in rubric_checks.items()]
)

display(rubric_df)

if all(rubric_checks.values()):
    print("Rubric checklist: PASS")
else:
    print("Rubric checklist: CHECK FAIL ITEMS BEFORE FINAL RUN")

### 7.2 Utility functions for plotting and metrics
Reuse common functions across model runs.

In [ ]:
def plot_training_history(history_dict, title: str):
    plt.figure(figsize=(9, 5))
    plt.plot(history_dict["loss"], label="Training Loss")
    plt.plot(history_dict["val_loss"], label="Validation Loss")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def evaluate_predictions(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return {"mse": mse, "mae": mae, "rmse": rmse, "r2": r2}

## 8. Train TensorFlow experiments

This section uses an explicit, rubric-compliant run plan (instead of brute-force full grid) to improve runtime stability.

### 8.1 Execute TensorFlow run plan
Each selected run trains for `EPOCHS` (100 in stable/full mode) and logs validation metrics.

#### 8.1.1 Initialize TensorFlow experiment trackers
Prepare containers for metrics, histories, model objects, and TensorBoard logs.

In [ ]:
tf_results = []
tf_histories_all = {}

best_tf_mae = float("inf")
best_tf_run = None
best_tf_row = None

best_model_path = Path("best_tf_model.keras")
if best_model_path.exists():
    best_model_path.unlink()

log_root = Path("tensorboard_logs")
log_root.mkdir(exist_ok=True)

#### 8.1.2 Run the TensorFlow training grid
Train each model architecture with each optimizer and loss setting, then evaluate on validation data.

### 8.1.2A Define a helper for one TensorFlow experiment


In [ ]:
def run_tensorflow_experiment(cfg: dict):
    model_name = cfg["model"]
    optimizer_name = cfg["optimizer"]
    loss_name = cfg["loss"]
    learning_rate = float(cfg["lr"])
    run_name = f"{model_name}_{optimizer_name}_{loss_name}_lr{learning_rate:g}"

    print(f"Training: {run_name}")

    model = MODEL_BUILDERS[model_name](X_train_scaled.shape[1])
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss=LOSS_MAP[loss_name],
        metrics=["mae"],
    )

    log_dir = log_root / run_name / time.strftime("%Y%m%d-%H%M%S")
    tb_callback = TensorBoard(log_dir=str(log_dir), histogram_freq=0, write_graph=False)

    start = time.time()
    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=TF_VERBOSE,
        callbacks=[tb_callback],
    )
    elapsed = time.time() - start

    y_pred = model.predict(X_val_scaled, verbose=0).reshape(-1)
    metrics = evaluate_predictions(y_val, y_pred)

    history_payload = {
        "loss": [float(v) for v in history.history["loss"]],
        "val_loss": [float(v) for v in history.history["val_loss"]],
    }

    row = {
        "run_name": run_name,
        "framework": "TensorFlow",
        "model": model_name,
        "optimizer": optimizer_name,
        "learning_rate": learning_rate,
        "loss": loss_name,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "train_time_sec": elapsed,
        "final_train_loss": float(history.history["loss"][-1]),
        "final_val_loss": float(history.history["val_loss"][-1]),
        "val_mse": float(metrics["mse"]),
        "val_mae": float(metrics["mae"]),
        "val_rmse": float(metrics["rmse"]),
        "val_r2": float(metrics["r2"]),
    }

    return run_name, row, history_payload, model


### 8.1.2B Execute the TensorFlow run plan


In [ ]:
print("CHECKPOINT 8.1.2: TensorFlow training plan start")

total_runs = len(EXPERIMENT_RUNS)
for run_idx, cfg in enumerate(EXPERIMENT_RUNS, start=1):
    run_name, row, history_payload, model = run_tensorflow_experiment(cfg)

    tf_histories_all[run_name] = history_payload
    tf_results.append(row)

    print(
        f"[{run_idx}/{total_runs}] done in {row['train_time_sec']:.1f}s | "
        f"val_mae={row['val_mae']:.4f} | val_mse={row['val_mse']:.4f}"
    )

    if row["val_mae"] < best_tf_mae:
        best_tf_mae = row["val_mae"]
        best_tf_run = run_name
        best_tf_row = row
        model.save(best_model_path, overwrite=True)

    del model
    tf.keras.backend.clear_session()
    gc.collect()


#### 8.1.3 Build and preview TensorFlow results table
Rank candidate runs by MAE and MSE to quickly identify top-performing settings.

In [ ]:
tf_results_df = pd.DataFrame(tf_results).sort_values(["val_mae", "val_mse"]).reset_index(drop=True)

all_history_runs = tf_results_df["run_name"].tolist()
top_history_runs = tf_results_df.head(min(TOP_CURVE_RUNS, len(tf_results_df)))["run_name"].tolist()
tf_histories = tf_histories_all

display_cols = [
    "run_name", "model", "optimizer", "learning_rate", "loss",
    "final_train_loss", "final_val_loss", "val_mse", "val_mae", "train_time_sec"
]

print("\nTop TensorFlow findings (sorted by val_mae, val_mse):")
display(tf_results_df[display_cols].head(TOP_FINDINGS_ROWS))


## 9. Plot training vs validation loss
Plot top runs by validation MAE for concise decision-making (summary mode), or all runs in detailed mode.

In [ ]:
print(f"Plotting {len(top_history_runs)} run(s) based on OUTPUT_MODE={OUTPUT_MODE}...")
for run_name in top_history_runs:
    plot_training_history(tf_histories[run_name], f"{run_name}: Training vs Validation Loss")

## 10. Select the best TensorFlow model and generate predictions

We select the best TensorFlow model using **lowest validation MAE**, then generate predictions and review performance.


### 10.1 Load the best TensorFlow model and compute validation predictions


In [ ]:
best_tf_row = tf_results_df.iloc[0]
best_tf_run = best_tf_row["run_name"]

best_tf_model = keras.models.load_model(best_model_path)
best_tf_preds = best_tf_model.predict(X_val_scaled, verbose=0).reshape(-1)
best_tf_metrics = evaluate_predictions(y_val, best_tf_preds)

print("Best TensorFlow run:", best_tf_run)
print(best_tf_row)


### 10.2 Preview actual and predicted trip durations


In [ ]:
pred_df = pd.DataFrame({
    "actual_trip_duration_min": y_val[:30],
    "predicted_trip_duration_min": best_tf_preds[:30],
})
display(pred_df)


### 10.3 Plot actual versus predicted trip duration


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_val, best_tf_preds, alpha=0.25)
min_v = float(min(y_val.min(), best_tf_preds.min()))
max_v = float(max(y_val.max(), best_tf_preds.max()))
plt.plot([min_v, max_v], [min_v, max_v], "r--", linewidth=1.5)
plt.title("Best TensorFlow Model: Actual vs Predicted")
plt.xlabel("Actual Trip Duration (min)")
plt.ylabel("Predicted Trip Duration (min)")
plt.tight_layout()
plt.show()


### 10.4 Inspect residual behavior and final validation metrics


In [ ]:
residuals = y_val - best_tf_preds
plt.figure(figsize=(9, 4))
plt.scatter(best_tf_preds, residuals, alpha=0.25)
plt.axhline(0, color="red", linestyle="--", linewidth=1.2)
plt.title("Best TensorFlow Model: Residual Plot")
plt.xlabel("Predicted Trip Duration (min)")
plt.ylabel("Residual (Actual - Predicted)")
plt.tight_layout()
plt.show()

print("\nBest TensorFlow validation metrics")
for k, v in best_tf_metrics.items():
    print(f"{k.upper()}: {v:.4f}")


## 11. TensorFlow model comparison summary

This table summarizes the best TensorFlow result for each architecture.


In [ ]:
best_per_arch = (
    tf_results_df
    .sort_values(["model", "val_mae", "val_mse"])
    .groupby("model", as_index=False)
    .first()
    .sort_values("val_mae")
    .reset_index(drop=True)
)

print("\nBest run per architecture (decision table):")
display(best_per_arch)

decision_dashboard = tf_results_df[[
    "run_name", "model", "optimizer", "learning_rate", "loss",
    "val_mae", "val_mse", "final_val_loss", "train_time_sec"
]].copy()

decision_dashboard["efficiency_score"] = decision_dashboard["val_mae"] * decision_dashboard["train_time_sec"]

print("\nDecision dashboard (top candidates):")
display(
    decision_dashboard
    .sort_values(["val_mae", "val_mse", "train_time_sec"])
    .head(TOP_FINDINGS_ROWS)
)


## 11.1 Overfitting diagnostics (TensorFlow)
Quantify generalization gap (`final_val_loss - final_train_loss`) to discuss overfitting risk, especially for deeper DNN runs.

In [ ]:
tf_overfit_df = tf_results_df.copy()
tf_overfit_df["generalization_gap"] = tf_overfit_df["final_val_loss"] - tf_overfit_df["final_train_loss"]

print("\nOverfitting diagnostics (largest generalization gaps):")
display(
    tf_overfit_df[[
        "run_name", "model", "optimizer", "learning_rate", "loss",
        "final_train_loss", "final_val_loss", "generalization_gap", "val_mae"
    ]].sort_values(["generalization_gap", "val_mae"], ascending=[False, True]).head(TOP_FINDINGS_ROWS)
)

plt.figure(figsize=(10, 4))
plot_df = tf_overfit_df.sort_values("generalization_gap", ascending=False).head(min(TOP_FINDINGS_ROWS, len(tf_overfit_df)))
plt.barh(plot_df["run_name"], plot_df["generalization_gap"], color="#d62728")
plt.gca().invert_yaxis()
plt.title("Top Generalization Gaps (Overfitting Risk)")
plt.xlabel("Validation Loss - Training Loss")
plt.tight_layout()
plt.show()


## 12. PyTorch extra credit: comparable DNN
PyTorch section is kept for extra credit and is enabled by default.

### 12.1 Build PyTorch DNN and data loaders
Use the same input features for a fair framework comparison.

In [ ]:
if RUN_TORCH_EXTRA:
    class TorchDNN(nn.Module):
        def __init__(self, input_dim: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 256),
                nn.ReLU(),
                nn.Dropout(0.30),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(0.30),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.20),
                nn.Linear(32, 1),
            )

        def forward(self, x):
            return self.net(x)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("PyTorch device:", device)

    X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
    X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
    y_val_t = torch.tensor(y_val.reshape(-1, 1), dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
else:
    print("PyTorch section skipped: RUN_TORCH_EXTRA=False")

## 13. Train the PyTorch DNN

To keep the extra credit comparable, we use:

- **Optimizer:** Adam
- **Loss:** MSELoss
- **Epochs:** 100


In [ ]:
if RUN_TORCH_EXTRA:
    torch_model = TorchDNN(X_train_scaled.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(torch_model.parameters(), lr=0.001)

    torch_history = {"train_loss": [], "val_loss": []}
    start = time.time()

### 13.1 Run PyTorch training loop
Train for `EPOCHS` and track train/validation loss history.

In [ ]:
if RUN_TORCH_EXTRA:
    print("CHECKPOINT 13.1: PyTorch training start")
    for epoch in range(TORCH_EPOCHS):
        torch_model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = torch_model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        torch_model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = torch_model(xb)
                loss = criterion(preds, yb)
                val_losses.append(loss.item())

        torch_history["train_loss"].append(float(np.mean(train_losses)))
        torch_history["val_loss"].append(float(np.mean(val_losses)))

        if (epoch + 1) % 20 == 0 or epoch == 0 or epoch == TORCH_EPOCHS - 1:
            print(f"Epoch {epoch+1}/{TORCH_EPOCHS} - train_loss={torch_history['train_loss'][-1]:.4f} - val_loss={torch_history['val_loss'][-1]:.4f}")

    torch_elapsed = time.time() - start

### 13.2 Evaluate PyTorch validation performance
Generate predictions and compute regression metrics.

In [ ]:
if RUN_TORCH_EXTRA:
    torch_model.eval()
    with torch.no_grad():
        torch_preds = torch_model(X_val_t.to(device)).cpu().numpy().reshape(-1)

    torch_metrics = evaluate_predictions(y_val, torch_preds)

    print("PyTorch training complete.")
    for k, v in torch_metrics.items():
        print(f"{k.upper()}: {v:.4f}")
    print(f"Training time (sec): {torch_elapsed:.2f}")

## 14. Plot PyTorch training vs validation loss


In [ ]:
if RUN_TORCH_EXTRA:
    plt.figure(figsize=(9, 5))
    plt.plot(torch_history["train_loss"], label="Training Loss")
    plt.plot(torch_history["val_loss"], label="Validation Loss")
    plt.title("PyTorch DNN: Training vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 15. Compare TensorFlow and PyTorch

We compare the best TensorFlow run against the PyTorch DNN on the same validation set.


In [ ]:
comparison_rows = [
    {
        "framework": "TensorFlow",
        "model": best_tf_row["model"],
        "optimizer": best_tf_row["optimizer"],
        "learning_rate": best_tf_row["learning_rate"],
        "loss": best_tf_row["loss"],
        "mse": best_tf_metrics["mse"],
        "mae": best_tf_metrics["mae"],
        "rmse": best_tf_metrics["rmse"],
        "r2": best_tf_metrics["r2"],
        "train_time_sec": float(best_tf_row["train_time_sec"]),
    }
]

if RUN_TORCH_EXTRA:
    comparison_rows.append(
        {
            "framework": "PyTorch",
            "model": "DNN",
            "optimizer": "Adam",
            "learning_rate": 0.001,
            "loss": "MSE",
            "mse": torch_metrics["mse"],
            "mae": torch_metrics["mae"],
            "rmse": torch_metrics["rmse"],
            "r2": torch_metrics["r2"],
            "train_time_sec": torch_elapsed,
        }
    )

framework_comparison_df = pd.DataFrame(comparison_rows).sort_values("mae").reset_index(drop=True)
display(framework_comparison_df)

## 16. Final conclusions

Use this section to write your final discussion clearly and directly:

- Which TensorFlow model performed best?
- Which optimizer worked best?
- Which loss function worked best?
- Did the DNN show overfitting?
- How did the PyTorch DNN compare with TensorFlow?


In [ ]:
print("Best TensorFlow run:", best_tf_run)
print("Best architecture:", best_tf_row["model"])
print("Best optimizer:", best_tf_row["optimizer"])
print("Best learning rate:", best_tf_row["learning_rate"])
print("Best loss function:", best_tf_row["loss"])

print("\nTensorFlow best metrics")
for k, v in best_tf_metrics.items():
    print(f"{k.upper()}: {v:.4f}")

if RUN_TORCH_EXTRA:
    print("\nPyTorch metrics")
    for k, v in torch_metrics.items():
        print(f"{k.upper()}: {v:.4f}")

print("\nRubric-aligned conclusion notes")
print("1. Time-aware 80/20 split was used (chronological split; no leakage).")
print("2. Features were scaled with StandardScaler before training.")
print("3. TensorFlow models compared: Linear, MLP, DNN with MSE/MAE and SGD/Adam/RMSProp.")
print("4. Each selected run used epoch=100 in stable/full modes (batch_size=32).")
print("5. Best model chosen by lowest validation MAE, then reviewed via predictions and residuals.")
print("6. Overfitting risk is discussed using generalization gap and loss curves.")